In [0]:
from pyspark import pipelines as dp
from pyspark.sql.functions import *

In [0]:
rules_customers = {
  "valid_customer_id": "customer_id IS NOT NULL",
  "valid_email": "COALESCE(TRIM(customer_email), '') <> ''",
  "valid_segment": "segment IN ('Home Office','Corporate','Consumer')",
  "valid_region": "region IN ('North','South','East','West','Central')"
}

quarantine_rules_customers = "NOT({0})".format(" AND ".join(rules_customers.values()))

@dp.table(name="globalmart_dev.silver.customers_quarantine")
def customers_quarantine():
  return (
    spark.readStream.table("customers_bronze_view")
    .withColumn("is_quarantined", expr(quarantine_rules_customers))
    .filter("is_quarantined = true")
    .withColumn(
      "issue_type",
      when(col("customer_email").isNull(), "missing_customer_email")
      .when(col("customer_id").isNull(), "missing_customer_id")
      .when(col("customer_name").isNull(), "missing_customer_name")
      .when(~col("segment").isin("Consumer", "Corporate", "Home Office"), "invalid_segment")
      .when(~col("region").isin("North", "South", "East", "West", "Central"), "invalid_region")
      .otherwise("other_issue")
    )
  )

@dp.table(
  name="globalmart_dev.silver.customers_silver",
  comment="cleaned table of the customers_bronze table"
)
@dp.expect_all(rules_customers)
def customers_silver():
  df = (
    spark.readStream.table("customers_bronze_view")
    .withColumn("is_quarantined", expr(quarantine_rules_customers))
    .filter("is_quarantined = false")
    .dropDuplicates(["customer_id"])
  )
  df = df.drop("_rescued_data", "_source_file_path", "_source_file_name", "_source_modified_time", "_ingested_at", "is_quarantined")
  return df

In [0]:
from pyspark import pipelines as dp
from pyspark.sql.functions import col, trim, initcap, when, expr, to_date

rules_orders = {
  "valid_order_id": "order_id IS NOT NULL",
  "valid_approved_date": "order_approved_at IS NOT NULL",
  "valid_carrier_delivered_date": "order_delivered_carrier_date IS NOT NULL",
  "valid_customer_delivery_date": "order_delivered_customer_date IS NOT NULL",
  "valid_estimated_date": "order_estimated_delivery_date IS NOT NULL",
  "valid_order_status": "order_status IN ('Processing','Shipped','Canceled','Delivered','Created','Invoiced','Unavailable')",
  "valid_ship_mode": "ship_mode IN ('Standard Class','Second Class','First Class','Same Day')"
}

quarantine_rules_orders = "NOT({0})".format(" AND ".join(rules_orders.values()))

@dp.table(name="globalmart_dev.silver.orders_quarantine")
def orders_quarantine():
  return (
    spark.readStream.table("orders_bronze_view")
    .withColumn("is_quarantined", expr(quarantine_rules_orders))
    .filter("is_quarantined = true")
    .withColumn(
    "issue_type",
    when(col("order_id").isNull(), "missing_order_id")
    .when(col("order_approved_at").isNull(), "missing_order_approved_at")
    .when(col("order_delivered_carrier_date").isNull(), "missing_order_delivered_carrier_date")
    .when(col("order_delivered_customer_date").isNull(), "missing_order_delivered_customer_date")
    .when(col("order_estimated_delivery_date").isNull(), "missing_order_estimated_delivery_date")
    .when(~col("order_status").isin('Processing','Shipped','Canceled','Delivered','Created','Invoiced','Unavailable'), "missing_order_status")
    .when(~col("ship_mode").isin('Standard Class','Second Class','First Class','Same Day'), "missing_ship_mode")
    .otherwise("other_issue")
)

    )
@dp.table(
  name="globalmart_dev.silver.orders_silver",
  comment="cleaned orders table"
)
@dp.expect_all(rules_orders)
def orders_silver():
  df = (
    spark.readStream.table("orders_bronze_view")


    # quarantine logic
    .withColumn("is_quarantined", expr(quarantine_rules_orders))
    .filter("is_quarantined = false")
  )
  df = df.drop("_rescued_data", "_source_file_path", "_source_file_name", "_source_modified_time", "_ingested_at", "is_quarantined")
  return df

In [0]:
rules_products = {
  "valid_product_id": "product_id IS NOT NULL",
  "valid_product_name": "product_name IS NOT NULL AND TRIM(product_name) != ''",
  "valid_category": "categories IS NOT NULL AND TRIM(categories) != ''",
}

quarantine_rules_products = "NOT({0})".format(" AND ".join(rules_products.values()))

# -------------------------
# QUARANTINE TABLE (BAD DATA)
# -------------------------
@dp.table(name="globalmart_dev.silver.products_quarantine")
def products_quarantine():
  return (
    spark.readStream.table("products")
    .withColumn("is_quarantined", expr(quarantine_rules_products))
    .filter("is_quarantined = true")
  )

# -------------------------
# SILVER TABLE (CLEAN DATA)
# -------------------------
@dp.table(
  name="globalmart_dev.silver.products_silver",
  comment="Cleaned and standardized products data"
)
@dp.expect_all(rules_products)
def products_silver():
  df = (
    spark.readStream.table("products")

    # Remove bad records
    .withColumn("is_quarantined", expr(quarantine_rules_products))
    .filter("is_quarantined = false")

    # -------------------------
    # STANDARDIZE CATEGORY
    # -------------------------
    .withColumn(
      "category",
      when(lower(col("categories")).like("%women%"), "Women")
      .when(lower(col("categories")).like("%men%"), "Men")
      .when(lower(col("categories")).like("%shoe%"), "Shoes")
      .when(lower(col("categories")).like("%clothing%"), "Clothing")
      .otherwise("Other")
    )

    # -------------------------
    # CLEAN PRODUCT NAME
    # -------------------------
  .withColumn("product_name", trim(col("product_name")))

  .withColumn(
  "sizes",
  when(col("sizes").isNull(), "One Size")
  .otherwise(col("sizes"))
  )

  .withColumn(
  "weight",
  when(col("weight").isNull(), 0)  # keep null
  .otherwise(col("weight"))
  )

  .withColumn(
  "dimension",
  when(col("dimension").isNull(), "Unknown")
  .otherwise(col("dimension"))
)

  .withColumn(
  "colors",
  when(col("colors").isNull(), "Not Specified")
  .otherwise(col("colors"))
)

  .withColumn("weight_value", regexp_extract(col("weight"), r"([0-9.]+)", 1).cast("double"))

  # -------------------------
  # Extract unit
  # -------------------------
  .withColumn("weight_unit", lower(regexp_extract(col("weight"), r"([a-zA-Z]+)", 1)))

  # -------------------------
  # Convert to KG
  # -------------------------
  .withColumn(
    "weight_kg",
    when(col("weight_unit").isin("kg"), col("weight_value"))
    .when(col("weight_unit").isin("g"), col("weight_value") / 1000)
    .when(col("weight_unit").isin("lb", "lbs", "pound", "pounds"), col("weight_value") * 0.453592)
    .otherwise(None)
  )
  .withColumn("dateAdded", to_date(col("dateAdded")))
  .withColumn("dateUpdated", to_date(col("dateUpdated")))

  # Optional: drop old column
  .drop("weight")
  )
  df = df.withColumns({"weight_kg": when(col("weight_kg").isNull(), 0).otherwise(col("weight_kg")), "manufacturer": when(col("manufacturer").isNull(), "Unknown").otherwise(col("manufacturer"))})
  
  df = df.withColumnRenamed("weight_kg", "weight")
  df = df.drop("_rescued_data", "_source_file_path", "_source_file_name", "_source_modified_time", "_ingested_at", "is_quarantined")
  return df

In [0]:
rules_returns = {
  "valid_return_date": "return_date IS NOT NULL",
  "valid_order_id" : "order_id is NOT NULL",
  "valid_amount": "refund_amount IS NOT NULL AND refund_amount > 0",
  "valid_return_reasons": "return_reason IS NOT NULL",
  "valid_return_reason": "return_reason != '?'"
  
}

quarantine_rules_returns = "NOT({0})".format(" AND ".join(rules_returns.values()))

@dp.table(name="globalmart_dev.silver.returns_quarantine")
def returns_quarantine():
  return (
    spark.readStream.table("returns_bronze_view")
    .withColumn("is_quarantined", expr(quarantine_rules_returns))
    .filter("is_quarantined = true")
  )

@dp.table(
  name="globalmart_dev.silver.returns_silver",
  comment="cleaned returns table"
)
@dp.expect_all(rules_returns)
def returns_silver():
  df = (
    spark.readStream.table("returns_bronze_view")

    # quarantine logic
    .withColumn("is_quarantined", expr(quarantine_rules_returns))
    .filter("is_quarantined = false")
  )
  df = df.drop("_rescued_data", "_source_file_path", "_source_file_name", "_source_modified_time", "_ingested_at", "is_quarantined")
  return df

In [0]:
rules_transactions = {
  "valid_order_id": "order_id IS NOT NULL",
  "valid_product_id": "product_id IS NOT NULL",
  "valid_sales": "sales IS NOT NULL AND sales > 0",
  "valid_quantity": "quantity IS NOT NULL AND quantity > 0",
  "valid_discount": "discount IS NOT NULL AND discount >= 0",
  "valid_profit": "profit IS NOT NULL AND profit >= 0",
  "valid_payment_type": "payment_type IS NOT NULL",
  "valid_installments": "payment_installments IS NOT NULL AND payment_installments > 0"
}

quarantine_rules_transactions = "NOT({0})".format(" AND ".join(rules_transactions.values()))

@dp.table(name="globalmart_dev.silver.transactions_quarantine")
def transactions_quarantine():
  return (
    spark.readStream.table("transactions_bronze_view")
    .withColumn("is_quarantined", expr(quarantine_rules_transactions))
    .filter("is_quarantined = true")
  )

  from pyspark.sql.functions import col, when, trim

@dp.table(
  name="globalmart_dev.silver.transactions_silver",
  comment="cleaned transactions table"
)
@dp.expect_all(rules_transactions)
def transactions_silver():
  df = (
    spark.readStream.table("transactions_bronze_view")

    # ✅ Clean numeric fields (remove '?', blanks)
    .withColumn("sales", when(col("sales") == "?", None).otherwise(trim(col("sales"))))
    .withColumn("quantity", when(col("quantity") == "?", None).otherwise(trim(col("quantity"))))
    .withColumn("discount", when(col("discount") == "?", None).otherwise(trim(col("discount"))))
    .withColumn("payment_installments", when(col("payment_installments") == "?", None).otherwise(trim(col("payment_installments"))))

    # ✅ Type casting
    .withColumn("sales", col("sales").cast("double"))
    .withColumn("quantity", col("quantity").cast("int"))
    .withColumn("discount", col("discount").cast("double"))
    .withColumn("payment_installments", col("payment_installments").cast("int"))

    # ✅ Optional: normalize payment type
    .withColumn("payment_type", trim(col("payment_type")))

    # quarantine logic
    .withColumn("is_quarantined", expr(quarantine_rules_transactions))
    .filter("is_quarantined = false")
  )
  df = df.drop("_rescued_data", "_source_file_path", "_source_file_name", "_source_modified_time", "_ingested_at", "is_quarantined")
  return df

In [0]:
rules_vendors = {
  "valid_vendor_id": "vendor_id IS NOT NULL AND TRIM(vendor_id) != ''",
  "valid_vendor_name": "vendor_name IS NOT NULL AND TRIM(vendor_name) != ''",
  "valid_vendor_id_format": "vendor_id RLIKE '^VEN[0-9]+$'"
}

quarantine_rules_vendors = "NOT({0})".format(" AND ".join(rules_vendors.values()))

@dp.table(name="globalmart_dev.silver.vendors_quarantine")
def vendors_quarantine():
  return (
    spark.readStream.table("vendors")
    .withColumn("is_quarantined", expr(quarantine_rules_vendors))
    .filter("is_quarantined = true")
  )

@dp.table(
  name="globalmart_dev.silver.vendors_silver",
  comment="cleaned vendors table"
)
@dp.expect_all(rules_vendors)
def vendors_silver():
  df = (
    spark.readStream.table("vendors")
    .withColumn("vendor_id", trim(col("vendor_id")))
    .withColumn("vendor_name", initcap(trim(col("vendor_name"))))
    .withColumn("is_quarantined", expr(quarantine_rules_vendors))
    .filter("is_quarantined = false")
    .dropDuplicates(["vendor_id"])
  )
  df = df.drop("_rescued_data", "_source_file_path", "_source_file_name", "_source_modified_time", "_ingested_at", "is_quarantined")
  return df